# 🏭 Day 01a: Singleton & Factory Patterns — Controlled Creation

---

## 🎯 What You'll Master Today
- Singleton: one instance, global access
- Factory Method: delegate object creation to subclasses
- Simple Factory: centralized object creation
- When to use (and when NOT to use) each pattern

---

## 🎭 The Analogies

**Singleton:** There's only ONE president of a country at any time. No matter how many people refer to "the president", they all refer to the same person.

**Factory:** Think of a pizza shop. You say "I want a Margherita" — you don't mix dough, sauce, cheese yourself. The shop (factory) decides HOW to make it.

---

In [ ]:
# ============================================
# 1. Singleton — Multiple Implementations
# ============================================

# Approach 1: Using __new__ (most Pythonic)
class DatabaseConnection:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance
    
    def __init__(self, host: str = "localhost", port: int = 5432):
        if self._initialized:
            return
        self.host = host
        self.port = port
        self._initialized = True
        print(f"  🔌 Connected to {host}:{port}")
    
    def query(self, sql: str):
        print(f"  📊 Executing: {sql}")


# Same instance every time!
db1 = DatabaseConnection("prod-db", 5432)
db2 = DatabaseConnection("other-db", 3306)  # __init__ skipped!

print(f"\ndb1 is db2? {db1 is db2}")  # True — same object
print(f"db1.host: {db1.host}")         # prod-db
print(f"db2.host: {db2.host}")         # prod-db (same object!)
print(f"id(db1) == id(db2)? {id(db1) == id(db2)}")  # True

In [ ]:
# ============================================
# 2. Singleton — Thread-Safe with Lock
# ============================================
import threading

class ThreadSafeSingleton:
    _instance = None
    _lock = threading.Lock()
    
    def __new__(cls):
        if cls._instance is None:
            with cls._lock:  # double-check locking
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance


# Verify thread safety
instances = []

def create_instance():
    instances.append(ThreadSafeSingleton())

threads = [threading.Thread(target=create_instance) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

# All 10 threads got the same instance
print(f"All same instance? {all(i is instances[0] for i in instances)}")
print(f"Unique instances: {len(set(id(i) for i in instances))}")

In [ ]:
# ============================================
# 3. Singleton — Metaclass Approach (Advanced)
# ============================================

class SingletonMeta(type):
    """Metaclass that makes any class a singleton."""
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class AppConfig(metaclass=SingletonMeta):
    def __init__(self):
        self.settings = {"debug": False, "db_url": "postgresql://..."}
    
    def get(self, key: str):
        return self.settings.get(key)


config1 = AppConfig()
config2 = AppConfig()
print(f"Same instance? {config1 is config2}")  # True
print(f"debug = {config1.get('debug')}")

# Change in one, visible in all
config1.settings["debug"] = True
print(f"debug via config2 = {config2.get('debug')}")  # True

```
╔═══════════════════════════════════════════════════════════════════╗
║  SINGLETON — When to Use & When NOT to                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  ✅ USE when:                    ❌ DON'T USE when:              ║
║  • Database connection pool      • You need multiple configs     ║
║  • Configuration/settings        • Testing (hard to mock!)       ║
║  • Logger                        • You want loose coupling       ║
║  • Cache manager                 • State causes hidden bugs      ║
║  • Thread pool                                                    ║
║                                                                   ║
║  ⚠️ Singletons are often considered an anti-pattern because they ║
║     introduce global state and make testing hard. Prefer DI.     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 4. Simple Factory — Centralized Creation
# ============================================
from abc import ABC, abstractmethod

class Notification(ABC):
    @abstractmethod
    def send(self, message: str) -> str:
        pass

class EmailNotification(Notification):
    def send(self, message: str) -> str:
        return f"📧 Email: {message}"

class SMSNotification(Notification):
    def send(self, message: str) -> str:
        return f"📱 SMS: {message}"

class PushNotification(Notification):
    def send(self, message: str) -> str:
        return f"🔔 Push: {message}"


# Simple Factory — a function or class that creates objects
class NotificationFactory:
    @staticmethod
    def create(notification_type: str) -> Notification:
        factories = {
            "email": EmailNotification,
            "sms": SMSNotification,
            "push": PushNotification,
        }
        creator = factories.get(notification_type)
        if not creator:
            raise ValueError(f"Unknown type: {notification_type}")
        return creator()


# Client doesn't know concrete classes!
for ntype in ["email", "sms", "push"]:
    notification = NotificationFactory.create(ntype)
    print(notification.send("Your order shipped!"))

In [ ]:
# ============================================
# 5. Factory Method Pattern — Subclass Decides
# ============================================
from abc import ABC, abstractmethod

class Document(ABC):
    @abstractmethod
    def render(self) -> str:
        pass

class PDFDocument(Document):
    def render(self) -> str:
        return "📄 Rendering PDF document"

class HTMLDocument(Document):
    def render(self) -> str:
        return "🌐 Rendering HTML document"

class MarkdownDocument(Document):
    def render(self) -> str:
        return "📝 Rendering Markdown document"


# Factory Method: the CREATOR defines an abstract factory method
class DocumentCreator(ABC):
    @abstractmethod
    def create_document(self) -> Document:
        """Factory method — subclasses decide WHICH document to create."""
        pass
    
    def generate_report(self, content: str) -> str:
        """Template: uses the factory method."""
        doc = self.create_document()
        return f"{doc.render()} — Content: {content}"


class PDFCreator(DocumentCreator):
    def create_document(self) -> Document:
        return PDFDocument()

class HTMLCreator(DocumentCreator):
    def create_document(self) -> Document:
        return HTMLDocument()

class MarkdownCreator(DocumentCreator):
    def create_document(self) -> Document:
        return MarkdownDocument()


# Client picks the creator, not the concrete document
creators = [PDFCreator(), HTMLCreator(), MarkdownCreator()]
for creator in creators:
    print(creator.generate_report("Q4 Sales Report"))

```
╔══════════════════╦═══════════════════════════════════════════════╗
║  Pattern          ║  When to Use                                 ║
╠══════════════════╬═══════════════════════════════════════════════╣
║  Simple Factory   ║  Few types, won't change often.              ║
║                   ║  Just a function/static method.              ║
╠══════════════════╬═══════════════════════════════════════════════╣
║  Factory Method   ║  Subclasses decide which class to create.   ║
║                   ║  Framework/library extensibility.            ║
╠══════════════════╬═══════════════════════════════════════════════╣
║  Abstract Factory ║  Families of related objects.                ║
║                   ║  (Next notebook!)                            ║
╚══════════════════╩═══════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is Singleton? | Only one instance of a class, globally accessible. Use __new__ or metaclass |
| 2 | How to make thread-safe Singleton? | Double-check locking with threading.Lock |
| 3 | Why are Singletons controversial? | Global state, hard to test/mock, hidden dependencies. Prefer DI |
| 4 | Simple Factory vs Factory Method? | Simple = static method returns object. Factory Method = subclass decides via abstract method |
| 5 | When to use Factory? | Object creation is complex, client shouldn't know concrete classes, or type is decided at runtime |
| 6 | Real-world Singleton examples? | DB connection pool, logger, cache, app config |
| 7 | Real-world Factory examples? | Notification system, document generators, payment processors |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Singleton = one instance (__new__, metaclass, module) │
## │  • Thread-safe = double-check locking with Lock          │
## │  • Singleton is often anti-pattern — prefer DI           │
## │  • Simple Factory = dict/if-else → return object         │
## │  • Factory Method = ABC with create method in subclass   │
## │  • Client depends on abstraction, factory decides type   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Abstract Factory & Builder** — creating families of objects